# Generation Corpus Counter (Full Study)

Counts every word the models actually produced across **all stages** of multi-stage prompting.
This is the *generation corpus* — distinct from the *final-answer corpus* in
`all_triplets_cache.csv` (which only contains the final synthesized report per video).

**File layouts handled:**

| Model    | Technique     | Source file(s) | Text-bearing fields                                                            |
|----------|---------------|----------------|---------------------------------------------------------------------------------|
| GPT      | Zero-Shot     | `zero_shot_checkpoint.json` | `batch_summaries` (list) + `final_analysis`                       |
| GPT      | Sequential    | `sequential_checkpoint.json` | `stages.stage1_batch_summaries` (list) + `stage1_*` thru `stage4_final_summary` |
| GPT      | Least-to-Most | `least_to_most_checkpoint.json` | `stages.level1_batch_summaries` (list) + `level1_*` thru `level5_final_report` |
| GPT      | ReAct         | `react_checkpoint.json` | `react_log.{thought1, action1_batch_obs, observation1, thought2, action2, thought3, final_answer}` |
| Claude   | (all 4)       | same checkpoint structures as GPT (assumed) | (same) |
| Gemini   | (all 4)       | per-video `*_complete_*.json` files | `least_to_most_results.{Step 1..N}.response` etc. |

**Output:**  `C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT\generation_corpus_full.json`

In [1]:
import os, json, re, glob
import pandas as pd

RESULTS_DIR = r'C:\Opeyemi\PROMPTS\RESULTS'
WC_DIR      = r'C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT'
os.makedirs(WC_DIR, exist_ok=True)

OUT_JSON  = os.path.join(WC_DIR, 'generation_corpus_full.json')
OUT_CSV   = os.path.join(WC_DIR, 'generation_corpus_full.csv')
OUT_TXT   = os.path.join(WC_DIR, 'generation_corpus_summary.txt')

# Word/sentence/claim counters
SENT_RE  = re.compile(r'[.!?]+(?:\s|$)')
CLAIM_RE = re.compile(r'(?:[.!?;]|--|—|\n\s*[-*•])')

def count_text(text: str):
    if not isinstance(text, str) or not text:
        return (0, 0, 0)
    words = len(text.split())
    sents = sum(1 for p in SENT_RE.split(text) if p.strip())
    claims = sum(1 for p in CLAIM_RE.split(text) if len(p.split()) >= 3)
    return (words, sents, claims)

def collect_strings_from(obj):
    """Recursively walk obj; yield every string found. Skips obvious non-text fields."""
    if isinstance(obj, str):
        yield obj
    elif isinstance(obj, list):
        for item in obj:
            yield from collect_strings_from(item)
    elif isinstance(obj, dict):
        for k, v in obj.items():
            # Skip obvious metadata keys
            if k in {'video_id','crime_type','prompting_technique','model','timestamp',
                     'frames_analyzed','total_batches','batch_size','complexity_level',
                     'frames_used','chunks_processed','frames_per_chunk','model_used',
                     'prompt'}:
                continue
            yield from collect_strings_from(v)

print(f'Results dir: {RESULTS_DIR}')
print(f'Output dir:  {WC_DIR}')


Results dir: C:\Opeyemi\PROMPTS\RESULTS
Output dir:  C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT


## 1. Walk GPT and Claude checkpoint files

In [2]:
# GPT and Claude store all videos for one technique in one big checkpoint JSON.
# Schema:
#   {
#     "completed_videos": [...],
#     "results": {video_id: {<technique-specific text fields>}, ...}
#   }
# We extract every string from each video's entry and tally words/sentences/claims.

# Discover all (model, technique) checkpoints
checkpoints = []
for model in ['GPT', 'CLAUDE']:
    model_dir = os.path.join(RESULTS_DIR, model)
    if not os.path.isdir(model_dir):
        print(f'  Skipping {model_dir} — not present')
        continue
    for tech in os.listdir(model_dir):
        tech_dir = os.path.join(model_dir, tech)
        if not os.path.isdir(tech_dir):
            continue
        # Find the checkpoint file (prefer _checkpoint.json over _summary_*.json)
        ckpts = glob.glob(os.path.join(tech_dir, '*checkpoint*.json'))
        if not ckpts:
            # fall back to any summary
            ckpts = sorted(glob.glob(os.path.join(tech_dir, '*summary*.json')))
            if ckpts:
                ckpts = [ckpts[-1]]  # latest summary
        if ckpts:
            checkpoints.append((model, tech, ckpts[0]))

print(f'Discovered {len(checkpoints)} checkpoint files:')
for m, t, p in checkpoints:
    size = os.path.getsize(p) / 1e6
    print(f'  {m:<7} {t:<16} {size:>6.1f} MB  {os.path.basename(p)}')


Discovered 8 checkpoint files:
  GPT     LEAST-TO-MOST      14.8 MB  least_to_most_checkpoint.json
  GPT     REACT              27.2 MB  react_checkpoint.json
  GPT     SEQUENTIAL         16.4 MB  sequential_checkpoint.json
  GPT     ZERO                5.3 MB  zero_shot_checkpoint.json
  CLAUDE  LEAST-TO-MOST     206.7 MB  least_to_most_checkpoint.json
  CLAUDE  REACT              43.0 MB  react_checkpoint.json
  CLAUDE  SEQUENTIAL         20.0 MB  sequential_checkpoint.json
  CLAUDE  ZERO               16.6 MB  zero_shot_checkpoint.json


In [3]:
# Now process each checkpoint
non_gemini_rows = []  # one row per video

for model, tech, path in checkpoints:
    print(f'\nProcessing {model} / {tech}...')
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    results = data.get('results', {})
    if not isinstance(results, dict):
        print(f'  WARNING: unexpected schema (no "results" dict). Skipping.')
        continue

    for video_id, entry in results.items():
        if not isinstance(entry, dict):
            continue
        # Walk every string in the entry, summing
        total_words, total_sents, total_claims = 0, 0, 0
        n_strings = 0
        total_chars = 0
        for s in collect_strings_from(entry):
            w, sn, c = count_text(s)
            total_words  += w
            total_sents  += sn
            total_claims += c
            n_strings    += 1
            total_chars  += len(s)

        # Normalize model name
        model_norm = 'Claude' if model == 'CLAUDE' else 'GPT'
        non_gemini_rows.append({
            'model': model_norm, 'technique': tech, 'video': video_id,
            'words': total_words, 'sentences': total_sents, 'claims': total_claims,
            'n_string_fields': n_strings, 'chars': total_chars,
        })

    print(f'  Processed {len(results):,} videos.')

print(f'\nTotal non-Gemini rows: {len(non_gemini_rows):,}')



Processing GPT / LEAST-TO-MOST...
  Processed 812 videos.

Processing GPT / REACT...
  Processed 812 videos.

Processing GPT / SEQUENTIAL...
  Processed 812 videos.

Processing GPT / ZERO...
  Processed 812 videos.

Processing CLAUDE / LEAST-TO-MOST...
  Processed 812 videos.

Processing CLAUDE / REACT...
  Processed 812 videos.

Processing CLAUDE / SEQUENTIAL...
  Processed 812 videos.

Processing CLAUDE / ZERO...
  Processed 812 videos.

Total non-Gemini rows: 6,496


## 2. Walk Gemini per-video files

In [4]:
# Gemini per-video files
#
# Three file-naming patterns exist in the RESULTS\GEMINI\ tree:
#   1) Multi-step techniques (LEAST-TO-MOST): one *_complete_*.json + several *_step{N}_*.json.
#      The complete file holds the full multi-step trace, so we read only the *_complete_*
#      file per video and skip step files (avoids double-counting).
#   2) Single-output techniques (ZERO-SHOT, REACT, SEQUENTIAL): one file per run, no _step or _complete
#      suffix. Pattern is e.g. *_zero_shot_analysis_<timestamp>.json.
#   3) Duplicate runs: some folders (notably ZERO-SHOT) have ~2 files per video because
#      the run was re-executed and both timestamps were saved. We dedupe by keeping
#      only the LATEST-timestamp file per video.

import re
from collections import defaultdict

VIDEO_ID_RE   = re.compile(r'^(.+?_x264)_')                    # captures up to "..._x264"
TIMESTAMP_RE  = re.compile(r'(\d{8}_\d{6})')                   # captures YYYYMMDD_HHMMSS

def gather_gemini_files(tech_dir):
    """Return one canonical file per video in tech_dir.
    
    Strategy:
      - Group all .json files (excluding checkpoints / step files) by video_id.
      - Within each group, prefer files with '_complete_' in the name (multi-step trace).
        If multiple complete files exist for one video, keep the latest timestamp.
      - If no complete file exists, keep the latest-timestamp file in the group.
    """
    all_files = glob.glob(os.path.join(tech_dir, '*.json'))
    by_video = defaultdict(list)
    for p in all_files:
        base = os.path.basename(p)
        # Skip checkpoint files and per-step files
        if 'checkpoint' in base.lower():
            continue
        if '_step' in base.lower() and '_complete_' not in base.lower():
            continue
        m = VIDEO_ID_RE.match(base)
        if not m:
            continue
        video_id = m.group(1)
        ts_match = TIMESTAMP_RE.search(base)
        ts = ts_match.group(1) if ts_match else ''
        is_complete = '_complete_' in base.lower()
        by_video[video_id].append((p, ts, is_complete))

    chosen = []
    for vid, candidates in by_video.items():
        # Prefer 'complete' files; otherwise keep latest timestamp
        complete_files = [c for c in candidates if c[2]]
        if complete_files:
            best = sorted(complete_files, key=lambda c: c[1], reverse=True)[0]
        else:
            best = sorted(candidates, key=lambda c: c[1], reverse=True)[0]
        chosen.append((vid, best[0]))
    return chosen


gemini_rows = []
gemini_dir = os.path.join(RESULTS_DIR, 'GEMINI')

if not os.path.isdir(gemini_dir):
    print(f'No Gemini dir at {gemini_dir} -- skipping.')
else:
    for tech in os.listdir(gemini_dir):
        tech_dir = os.path.join(gemini_dir, tech)
        if not os.path.isdir(tech_dir):
            continue

        n_total = len(glob.glob(os.path.join(tech_dir, '*.json')))
        chosen  = gather_gemini_files(tech_dir)
        print(f'  Gemini / {tech}: {n_total:>5} files on disk -> {len(chosen):>4} kept after dedupe')

        for video_id, path in chosen:
            try:
                with open(path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
            except Exception as e:
                print(f'    ERROR reading {path}: {e}')
                continue

            total_words, total_sents, total_claims = 0, 0, 0
            n_strings = 0
            total_chars = 0
            for s in collect_strings_from(data):
                w, sn, c = count_text(s)
                total_words += w
                total_sents += sn
                total_claims += c
                n_strings += 1
                total_chars += len(s)

            gemini_rows.append({
                'model': 'Gemini', 'technique': tech, 'video': video_id,
                'words': total_words, 'sentences': total_sents, 'claims': total_claims,
                'n_string_fields': n_strings, 'chars': total_chars,
            })

print(f'\nTotal Gemini rows after dedupe: {len(gemini_rows):,}')


  Gemini / LEAST-TO-MOST:  7370 files on disk ->  811 kept after dedupe
  Gemini / REACT:  6911 files on disk ->  812 kept after dedupe
  Gemini / SEQUENTIAL:  5716 files on disk ->  812 kept after dedupe
  Gemini / ZERO:  1568 files on disk ->  812 kept after dedupe

Total Gemini rows after dedupe: 3,247


## 3. Combine, save, and summarize

In [5]:
# Combine all rows
all_rows = non_gemini_rows + gemini_rows
df = pd.DataFrame(all_rows)
print(f'Total per-video rows across all models/techniques: {len(df):,}')

# Normalize technique labels for grouping (e.g. "ZERO" vs "Zero-Shot" vs "ZERO-SHOT")
TECH_NORM = {
    'ZERO': 'Zero-Shot', 'ZERO-SHOT': 'Zero-Shot',
    'SEQUENTIAL': 'Sequential',
    'LEAST-TO-MOST': 'Least-to-Most',
    'REACT': 'ReAct',
}
df['technique'] = df['technique'].str.upper().map(TECH_NORM).fillna(df['technique'])

# Per-(model, technique) totals
tech_details = {}
for (model, tech), grp in df.groupby(['model', 'technique']):
    tech_details.setdefault(model, {})[tech] = {
        'experiments': int(len(grp)),
        'words':       int(grp['words'].sum()),
        'sentences':   int(grp['sentences'].sum()),
        'claims':      int(grp['claims'].sum()),
        'mean_words':  round(float(grp['words'].mean()), 1),
    }

# Per-model totals
model_results = {}
for model, grp in df.groupby('model'):
    model_results[model] = {
        'experiments': int(len(grp)),
        'words':       int(grp['words'].sum()),
        'sentences':   int(grp['sentences'].sum()),
        'claims':      int(grp['claims'].sum()),
    }

grand = {
    'experiments': int(len(df)),
    'words':       int(df['words'].sum()),
    'sentences':   int(df['sentences'].sum()),
    'claims':      int(df['claims'].sum()),
}

corpus = {
    'source': RESULTS_DIR,
    'description': 'Generation corpus: every word produced during multi-stage prompting (all stages summed).',
    'model_results': model_results,
    'grand_totals': grand,
    'technique_details': tech_details,
}

with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(corpus, f, indent=2)
print(f'\nSaved JSON: {OUT_JSON}')

df.to_csv(OUT_CSV, index=False)
print(f'Saved CSV:  {OUT_CSV}')

# Pretty summary
lines = []
lines.append('=' * 72)
lines.append('GENERATION CORPUS — full study')
lines.append('=' * 72)
lines.append('')
lines.append(f'Total per-video records: {grand["experiments"]:,}')
lines.append(f'Total words:             {grand["words"]:>15,}')
lines.append(f'Total sentences:         {grand["sentences"]:>15,}')
lines.append(f'Total claims:            {grand["claims"]:>15,}')
lines.append('')
lines.append('--- Per-model totals ---')
for m, d in model_results.items():
    lines.append(f'  {m:<10} exps={d["experiments"]:>5,}  words={d["words"]:>14,}  '
                 f'sents={d["sentences"]:>10,}  claims={d["claims"]:>10,}')
lines.append('')
lines.append('--- Per (model, technique) ---')
lines.append(f'{"Model":<10} {"Technique":<16} {"Exps":>6} {"Words":>14} {"MeanWords":>10} {"Claims":>10}')
lines.append('-' * 72)
for model, techs in tech_details.items():
    for tech, d in techs.items():
        lines.append(f'{model:<10} {tech:<16} {d["experiments"]:>6,} {d["words"]:>14,} '
                     f'{d["mean_words"]:>10.1f} {d["claims"]:>10,}')
summary = '\n'.join(lines)
print('\n' + summary)
with open(OUT_TXT, 'w', encoding='utf-8') as f:
    f.write(summary)
print(f'\nSaved TXT: {OUT_TXT}')


Total per-video rows across all models/techniques: 9,743

Saved JSON: C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT\generation_corpus_full.json
Saved CSV:  C:\Opeyemi\PROMPTS\WORD-COUNT-RESULT\generation_corpus_full.csv

GENERATION CORPUS — full study

Total per-video records: 9,743
Total words:                  64,073,675
Total sentences:               2,681,835
Total claims:                  4,884,153

--- Per-model totals ---
  Claude     exps=3,248  words=    37,599,003  sents= 1,158,181  claims= 3,205,566
  GPT        exps=3,248  words=     8,941,270  sents=   555,071  claims=   642,289
  Gemini     exps=3,247  words=    17,533,402  sents=   968,583  claims= 1,036,298

--- Per (model, technique) ---
Model      Technique          Exps          Words  MeanWords     Claims
------------------------------------------------------------------------
Claude     Least-to-Most       812     26,606,855    32767.1  2,261,923
Claude     ReAct               812      5,962,808     7343.4    526,456
Claude